In [2]:
import ee
import geemap

ee.Authenticate()

ee.Initialize()

print("Google Earth Engine Connected!")

Enter verification code:  4/1AXEQxIDQ59WIrOjv4An_dOvnqWq5GgpqborQOYnMDNvV1bK4bVLMjZmL9mQ



Successfully saved authorization token.
Google Earth Engine Connected!


In [2]:
import pandas as pd

ee.Initialize(project='ee-chowdharypranav2005')

print("✅ Earth Engine Connected")

NameError: name 'ee' is not defined

In [3]:
districts = pd.read_csv(
    "../data/raw/outbreaks/district_master.csv"
)

districts.head()

,State,District,Latitude,Longitude,Forest_Type,Elevation_Category
0,Karnataka,Shivamogga,13.9299,75.5681,Tropical Moist Deciduous,Medium
1,Karnataka,Uttara Kannada,14.7937,74.6869,Tropical Evergreen,Medium
2,Karnataka,Chikkamagaluru,13.3161,75.7720,Tropical Evergreen,High
3,Karnataka,Kodagu,12.4244,75.7382,Tropical Evergreen,High
4,Kerala,Wayanad,11.6854,76.1320,Tropical Evergreen,High


In [5]:
district = districts.iloc[0]

lat = district["Latitude"]
lon = district["Longitude"]

year = 2015

point = ee.Geometry.Point([lon, lat])
region = point.buffer(50000)  # 50 km buffer

In [7]:
sentinel = (
    ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
    .filterBounds(region)
    .filterDate(f"{year}-01-01", f"{year}-12-31")
    .median()
)

In [8]:
ndvi = sentinel.normalizedDifference(
    ["B8", "B4"]
).rename("NDVI")

In [9]:
ndvi_mean = ndvi.reduceRegion(
    reducer=ee.Reducer.mean(),
    geometry=region,
    scale=30,
    maxPixels=1e9
)

print(ndvi_mean.getInfo())

{'NDVI': 0.5868659691704521}


In [10]:
Map = geemap.Map(center=[lat, lon], zoom=9)

ndvi_vis = {
    "min": 0,
    "max": 1,
    "palette": ["white", "yellow", "green"]
}

Map.addLayer(ndvi, ndvi_vis, f"NDVI {year}")

Map

Map(center=[np.float64(13.9299), np.float64(75.5681)], controls=(WidgetControl(options=['position', 'transpare…

In [17]:
def extract_ndvi(lat, lon, year, buffer_radius=...):
   
    point = ee.Geometry.Point([lon, lat])
    
    collection = (
        ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED") # or "COPERNICUS/S2_SR"
        .filterDate(f"{year}-01-01", f"{year}-12-31")
        .filterBounds(point)
        .select(["B4", "B8"]) 
    )
    
    return ndvi_mean.get("NDVI").getInfo()

In [18]:
value = extract_ndvi(
    lat=13.9299,
    lon=75.5681,
    year=2015
)

print(value)

0.5868659691704522


In [26]:
years = range(2015, 2025)

records = []

for _, district in districts.iterrows():

    for year in range(2015, 2025):

        print(f"Processing {district['District']} - {year}")

        ndvi = extract_ndvi(
            district["Latitude"],
            district["Longitude"],
            year
        )

        records.append({
            "State": district["State"],
            "District": district["District"],
            "Year": year,
            "NDVI": ndvi
        })

ndvi_dataset = pd.DataFrame(records)

ndvi_dataset.head()

Processing Shivamogga - 2015
Processing Shivamogga - 2016
Processing Shivamogga - 2017
Processing Shivamogga - 2018
Processing Shivamogga - 2019
Processing Shivamogga - 2020
Processing Shivamogga - 2021
Processing Shivamogga - 2022
Processing Shivamogga - 2023
Processing Shivamogga - 2024
Processing Uttara Kannada - 2015
Processing Uttara Kannada - 2016
Processing Uttara Kannada - 2017
Processing Uttara Kannada - 2018
Processing Uttara Kannada - 2019
Processing Uttara Kannada - 2020
Processing Uttara Kannada - 2021
Processing Uttara Kannada - 2022
Processing Uttara Kannada - 2023
Processing Uttara Kannada - 2024
Processing Chikkamagaluru - 2015
Processing Chikkamagaluru - 2016
Processing Chikkamagaluru - 2017
Processing Chikkamagaluru - 2018
Processing Chikkamagaluru - 2019
Processing Chikkamagaluru - 2020
Processing Chikkamagaluru - 2021
Processing Chikkamagaluru - 2022
Processing Chikkamagaluru - 2023
Processing Chikkamagaluru - 2024
Processing Kodagu - 2015
Processing Kodagu - 2016


,State,District,Year,NDVI
0,Karnataka,Shivamogga,2015,0.586866
1,Karnataka,Shivamogga,2016,0.586866
2,Karnataka,Shivamogga,2017,0.586866
3,Karnataka,Shivamogga,2018,0.586866
4,Karnataka,Shivamogga,2019,0.586866


In [4]:
ndvi_dataset.to_csv(
    "../data/processed/ndvi_western_ghats_2015_2024.csv",
    index=False
)

print("NDVI Dataset Saved!")

NameError: name 'ndvi_dataset' is not defined

In [28]:
print(ndvi_dataset.shape)

print(ndvi_dataset.isnull().sum())

print(ndvi_dataset.describe())

(90, 4)
State       0
District    0
Year        0
NDVI        0
dtype: int64
              Year          NDVI
count    90.000000  9.000000e+01
mean   2019.500000  5.868660e-01
std       2.888373  1.116443e-16
min    2015.000000  5.868660e-01
25%    2017.000000  5.868660e-01
50%    2019.500000  5.868660e-01
75%    2022.000000  5.868660e-01
max    2024.000000  5.868660e-01
